# 01 — Validate Synthetic Raw Data

## Purpose and process

This notebook confirms that the supplied synthetic CSV files are suitable for warehouse loading.

**Logic chain:** locate files → confirm schemas → validate keys and domains → check cross-file relationships → summarise daily coverage.


## 1. Locate the project and source files

The project root is detected from the current working directory so the notebook does not depend on a personal computer path.


In [1]:
from pathlib import Path
import re

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw" / "users.csv").exists():
            return candidate
    raise FileNotFoundError("Project root could not be located.")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"

event_files = sorted(RAW_DIR.glob("daily_events_*.csv"))
order_files = sorted(RAW_DIR.glob("daily_orders_*.csv"))

assert len(event_files) == 7, f"Expected 7 event files; found {len(event_files)}."
assert len(order_files) == 7, f"Expected 7 order files; found {len(order_files)}."

print(f"Project root: {PROJECT_ROOT.name}")
print(f"Raw data directory: data/raw")
print(f"Event files: {len(event_files)}")
print(f"Order files: {len(order_files)}")


Project root: customer-funnel-ltv-data-warehouse
Raw data directory: data/raw
Event files: 7
Order files: 7


## 2. Load and combine the files

Each daily file receives a source date before the seven event files and seven order files are combined.


In [2]:
def source_date(path: Path) -> pd.Timestamp:
    match = re.search(r"(\d{4}-\d{2}-\d{2})", path.name)
    if not match:
        raise ValueError(f"Date not found in filename: {path.name}")
    return pd.Timestamp(match.group(1))


users = pd.read_csv(RAW_DIR / "users.csv")

event_frames = []
for path in event_files:
    frame = pd.read_csv(path)
    frame["source_date"] = source_date(path)
    frame["source_file"] = path.name
    event_frames.append(frame)
events = pd.concat(event_frames, ignore_index=True)

order_frames = []
for path in order_files:
    frame = pd.read_csv(path)
    frame["source_date"] = source_date(path)
    frame["source_file"] = path.name
    order_frames.append(frame)
orders = pd.concat(order_frames, ignore_index=True)

users["registration_date"] = pd.to_datetime(users["registration_date"], errors="coerce")
events["event_timestamp"] = pd.to_datetime(
    events["event_timestamp"], errors="coerce", format="mixed"
)
orders["order_timestamp"] = pd.to_datetime(
    orders["order_timestamp"], errors="coerce", format="mixed"
)

print(f"Users loaded: {len(users):,}")
print(f"Events loaded: {len(events):,}")
print(f"Orders loaded: {len(orders):,}")


Users loaded: 3,000
Events loaded: 5,181
Orders loaded: 283


## 3. Validate schemas, keys, and domains

These checks fail immediately if a required field is missing, a business identifier is duplicated, or a value falls outside the expected synthetic domains.


In [3]:
EXPECTED_USERS = {"user_id", "registration_date", "region", "age_group"}
EXPECTED_EVENTS = {
    "event_id", "user_id", "event_type", "event_timestamp",
    "device", "channel", "session_id"
}
EXPECTED_ORDERS = {
    "order_id", "user_id", "order_timestamp", "amount",
    "currency", "payment_method", "status"
}

VALID_REGIONS = {"ACT", "NSW", "NT", "QLD", "SA", "TAS", "VIC", "WA"}
VALID_AGE_GROUPS = {"18-24", "25-34", "35-44", "45-54", "55-64", "65+"}
VALID_EVENTS = {"view", "click", "purchase"}
VALID_DEVICES = {"desktop", "mobile", "tablet"}
VALID_CHANNELS = {"direct", "email", "referral", "search", "social"}
VALID_PAYMENT_METHODS = {"bank_transfer", "card", "paypal"}
VALID_STATUSES = {"paid", "pending", "refunded"}


def check_schema(actual: set[str], expected: set[str], name: str) -> None:
    missing = expected - actual
    assert not missing, f"{name} is missing required columns: {sorted(missing)}"
    print(f"PASS — {name} includes all required columns.")


check_schema(set(users.columns), EXPECTED_USERS, "users.csv")
check_schema(set(events.columns), EXPECTED_EVENTS, "event files")
check_schema(set(orders.columns), EXPECTED_ORDERS, "order files")

assert users["user_id"].notna().all() and users["user_id"].is_unique
assert events["event_id"].notna().all() and events["event_id"].is_unique
assert orders["order_id"].notna().all() and orders["order_id"].is_unique
print("PASS — user, event, and order business identifiers are non-null and unique.")

assert set(users["region"].dropna()) <= VALID_REGIONS
assert set(users["age_group"].dropna()) <= VALID_AGE_GROUPS
assert set(events["event_type"].dropna()) <= VALID_EVENTS
assert set(events["device"].dropna()) <= VALID_DEVICES
assert set(events["channel"].dropna()) <= VALID_CHANNELS
assert set(orders["payment_method"].dropna()) <= VALID_PAYMENT_METHODS
assert set(orders["status"].dropna()) <= VALID_STATUSES
assert set(orders["currency"].dropna()) == {"AUD"}
assert (orders["amount"] > 0).all()
print("PASS — categorical domains, currency, and positive order amounts are valid.")

assert users["registration_date"].notna().all()
assert events["event_timestamp"].notna().all()
assert orders["order_timestamp"].notna().all()
print("PASS — registration, event, and order timestamps are parseable.")


PASS — users.csv includes all required columns.
PASS — event files includes all required columns.
PASS — order files includes all required columns.
PASS — user, event, and order business identifiers are non-null and unique.
PASS — categorical domains, currency, and positive order amounts are valid.
PASS — registration, event, and order timestamps are parseable.


## 4. Validate relationships and daily consistency

The checks confirm that every event and order maps to a supplied user, timestamps agree with their daily source files, and funnel-stage user sets remain nested within each day.


In [4]:
user_ids = set(users["user_id"])
assert set(events["user_id"]) <= user_ids
assert set(orders["user_id"]) <= user_ids
print("PASS — every event and order maps to a supplied user.")

assert (events["event_timestamp"].dt.normalize() == events["source_date"]).all()
assert (orders["order_timestamp"].dt.normalize() == orders["source_date"]).all()
print("PASS — event and order timestamps match their source-file dates.")

for day, daily_events in events.groupby("source_date"):
    stage_users = {
        stage: set(daily_events.loc[daily_events["event_type"] == stage, "user_id"])
        for stage in ("view", "click", "purchase")
    }
    daily_paid_users = set(
        orders.loc[
            (orders["source_date"] == day) & (orders["status"] == "paid"),
            "user_id",
        ]
    )
    assert stage_users["click"] <= stage_users["view"]
    assert stage_users["purchase"] <= stage_users["click"]
    assert daily_paid_users <= stage_users["click"]

print("PASS — daily click, purchase-event, and paid-order users follow the funnel sequence.")


PASS — every event and order maps to a supplied user.
PASS — event and order timestamps match their source-file dates.
PASS — daily click, purchase-event, and paid-order users follow the funnel sequence.


## 5. Summarise validated coverage

This final table provides an execution record for the seven-day source period.


In [5]:
daily_events = (
    events.groupby("source_date")
    .agg(
        event_rows=("event_id", "size"),
        view_users=("user_id", lambda s: s[events.loc[s.index, "event_type"].eq("view")].nunique()),
        click_users=("user_id", lambda s: s[events.loc[s.index, "event_type"].eq("click")].nunique()),
        purchase_event_users=("user_id", lambda s: s[events.loc[s.index, "event_type"].eq("purchase")].nunique()),
    )
    .reset_index()
)

daily_orders = (
    orders.groupby("source_date")
    .agg(
        order_rows=("order_id", "size"),
        paid_orders=("status", lambda s: s.eq("paid").sum()),
    )
    .reset_index()
)

daily_summary = daily_events.merge(daily_orders, on="source_date", how="outer")
daily_summary["source_date"] = daily_summary["source_date"].dt.strftime("%Y-%m-%d")

print(daily_summary.to_string(index=False))
print("\nVALIDATION COMPLETE — all required checks passed.")


source_date  event_rows  view_users  click_users  purchase_event_users  order_rows  paid_orders
 2025-10-01         813         544          223                    46          46           44
 2025-10-02         788         535          213                    40          40           34
 2025-10-03         762         527          195                    40          40           34
 2025-10-04         580         410          145                    25          25           22
 2025-10-05         622         411          176                    35          35           29
 2025-10-06         814         547          222                    45          45           43
 2025-10-07         802         541          209                    52          52           47

VALIDATION COMPLETE — all required checks passed.
